# Bài 13 - Bộ nhớ của Tác nhân


## Cài đặt

Sổ tay này trình bày cách xây dựng một đại lý đặt chuyến du lịch với **bộ nhớ bền vững** sử dụng **Khung Đại lý Microsoft** (MAF).

Bạn sẽ tìm hiểu cách các loại bộ nhớ đại lý khác nhau — làm việc, ngắn hạn và dài hạn — ảnh hưởng đến cách đại lý lưu giữ và sử dụng thông tin qua các cuộc trò chuyện.

**Yêu cầu trước:**
- Một dự án Microsoft Foundry với một mô hình trò chuyện đã được triển khai (ví dụ `gpt-4.1-mini`).
- Đăng nhập bằng Azure CLI — chạy `az login` trong terminal của bạn.
- `AZURE_AI_PROJECT_ENDPOINT` — điểm cuối dự án Microsoft Foundry của bạn.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — tên mô hình đã triển khai của bạn.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import json
import dotenv
from typing import Annotated
from datetime import datetime

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Các loại Bộ nhớ của Agent

Các agent AI có thể tận dụng các loại bộ nhớ khác nhau, mỗi loại phục vụ một mục đích riêng biệt:

### Bộ nhớ làm việc
Chuỗi cuộc trò chuyện — các tin nhắn trao đổi trong một phiên làm việc. Agent có thể tham khảo lại các tin nhắn trước đó trong cùng một chuỗi để duy trì sự nhất quán. Trong MAF, điều này được tạo ra thông qua **`agent.create_session()`**, trả về một `AgentSession`.

### Bộ nhớ ngắn hạn
Thông tin tồn tại trong suốt thời gian của một nhiệm vụ hoặc phiên làm việc nhưng không được lưu trữ vĩnh viễn. Ví dụ, agent có thể tích lũy các sự thật trong một cuộc trò chuyện lập kế hoạch nhiều lượt và sử dụng chúng để tạo ra hành trình cuối cùng.

### Bộ nhớ dài hạn
Các sở thích và sự thật tồn tại **trong các phiên làm việc liên tiếp**. Người dùng quay lại không nên phải lặp lại các hạn chế ăn uống hoặc phong cách đi lại của họ. Bộ nhớ dài hạn thường được hỗ trợ bởi một kho lưu trữ bên ngoài — cơ sở dữ liệu, tệp hoặc chỉ mục vector — và được hiển thị cho agent thông qua các công cụ.


## Bộ Nhớ Làm Việc với Các Phiên

Hình thức đơn giản nhất của bộ nhớ là phiên trò chuyện. Khi bạn truyền cùng một phiên (được tạo bằng `agent.create_session()`) vào các lần gọi `agent.run()` kế tiếp, agent sẽ thấy toàn bộ lịch sử của cuộc trò chuyện đó và có thể nhớ lại các chi tiết trước đó.

Hãy tạo một agent du lịch và trình bày bộ nhớ làm việc.


In [ ]:
agent = client.as_agent(
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Đại lý đã nhớ đúng ngân sách vì cả hai tin nhắn đều dùng chung một phiên làm việc. Đây là **bộ nhớ làm việc** — nó chỉ tồn tại trong suốt thời gian của phiên làm việc.

### Điều gì sẽ xảy ra với một luồng mới?

Nếu chúng ta tạo một phiên làm việc **mới**, đại lý sẽ không còn nhớ cuộc trò chuyện trước đó:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Mẫu bộ nhớ dài hạn

Để ghi nhớ sở thích của người dùng **qua các phiên làm việc**, chúng ta cần một kho lưu trữ lâu dài tồn tại bên ngoài chuỗi hội thoại. Tác nhân truy cập kho lưu trữ này thông qua **công cụ** — các hàm mà nó có thể gọi để lưu và truy xuất thông tin.

Dưới đây chúng tôi triển khai một kho lưu trữ sở thích đơn giản trong bộ nhớ (trong môi trường sản xuất bạn sẽ dùng cơ sở dữ liệu hoặc chỉ mục vector) và cung cấp nó dưới dạng các công cụ mà tác nhân có thể sử dụng.

### Kiến trúc
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Tình huống 1 — Người dùng lần đầu tiên đặt chuyến đi kỷ niệm

Sarah ghé thăm lần đầu tiên. Đại lý nên lưu trữ sở thích của cô ấy thông qua các công cụ và sử dụng chúng để đề xuất các khách sạn.


In [ ]:
travel_agent = client.as_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Tình huống 2 — Sarah trở lại sau vài tuần

Sarah bắt đầu một **luồng thảo luận hoàn toàn mới** (mô phỏng một phiên làm việc mới). Bộ nhớ làm việc thì trống, nhưng kho lưu trữ sở thích lâu dài vẫn còn thông tin của cô ấy. Đại lý nên truy xuất nó và sử dụng để cá nhân hóa các đề xuất.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Tóm tắt

Trong bài học này, bạn đã học về ba loại bộ nhớ của đại lý và cách triển khai chúng với Microsoft Agent Framework:

| Loại bộ nhớ | Cơ chế MAF | Thời gian tồn tại |
|---|---|---|
| **Làm việc** | `agent.create_session()` | Một cuộc trò chuyện đơn |
| **Ngắn hạn** | Ngữ cảnh tích lũy trong một luồng | Một nhiệm vụ / phiên làm việc |
| **Dài hạn** | Kho lưu trữ bên ngoài truy cập qua các hàm `@tool` | Qua nhiều phiên làm việc |

### Những điểm chính cần nhớ
1. **`agent.create_session()`** cung cấp bộ nhớ làm việc — đại lý thấy toàn bộ lịch sử cuộc trò chuyện trong một phiên làm việc.
2. **Các phiên làm việc mới mất ngữ cảnh** — nếu không có bộ nhớ dài hạn, đại lý không thể nhớ các cuộc trò chuyện trước đó.
3. **Các hàm `@tool`** kết nối khoảng cách — chúng cho phép đại lý lưu và truy xuất thông tin từ kho lưu trữ tồn tại lâu dài.
4. **Cá nhân hóa được cải thiện theo thời gian** — càng lưu nhiều sở thích thì đề xuất của đại lý càng chính xác hơn.

### Ứng dụng trong thực tế
- **Dịch vụ khách hàng**: Nhớ lịch sử và sở thích của khách hàng
- **Trợ lý cá nhân**: Duy trì ngữ cảnh qua nhiều ngày hoặc tuần
- **Chăm sóc sức khỏe**: Theo dõi thông tin và sở thích của bệnh nhân
- **Thương mại điện tử**: Mua sắm cá nhân hóa dựa trên lịch sử

### Các bước tiếp theo
- Thay thế dict trong bộ nhớ bằng cơ sở dữ liệu hoặc kho vector (ví dụ Azure AI Search)
- Thêm tính năng hết hạn bộ nhớ cho thông tin nhạy cảm theo thời gian
- Xây dựng hệ thống đa đại lý với bộ nhớ dùng chung
- Khám phá sổ tay Cognee cho bộ nhớ được hỗ trợ bởi biểu đồ tri thức


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
